In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path, index_col=0)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df
# Function to split the name column and create new columns
def split_name_column(name):
    parts = name.split('_')
    parameters = parts[-1].replace('.qasm', '').strip('[]')
    position = parts[5].replace('P', '')
    qubit = parts[6].replace('Q', '')
    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters


# Example usage:
folder_path = './results'
df = read_and_merge_csv_files(folder_path)
# Apply the function to the name column and create new columns
df[['Algorithm', 'Qubits_number',  'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

# Drop the original name column if desired
df = df.drop(columns=['Name'])
df

,Input,Ideal_chisquare,Noisy_chisquare,Ideal_hellinger,Noisy_hellinger,Ideal_trace,Noisy_trace,Ideal_fidelity,Noisy_fidelity,Killed_IC,...,Killed_NT,Killed_IF,Killed_NF,Algorithm,Qubits_number,Operator,Gate,Position,Qubits,Params
0,PureState_0,1.320577e-35,3.859073e-45,0.126210,0.137953,0.117157,0.116919,0.859411,0.860433,True,...,True,True,True,ae,2,Add,cp,2,0,0.7853981633974483
1,PureState_1,2.443172e-24,3.536481e-30,0.105668,0.116247,0.117157,0.116970,0.859411,0.860332,True,...,True,True,True,ae,2,Add,cp,2,0,0.7853981633974483
2,PureState_2,7.068121e-09,5.816753e-09,0.072484,0.070934,0.074981,0.074580,0.947279,0.947566,True,...,True,True,True,ae,2,Add,cp,2,0,0.7853981633974483
3,PureState_3,1.032762e-03,2.855189e-05,0.043873,0.053360,0.074981,0.074549,0.947279,0.947588,True,...,True,True,True,ae,2,Add,cp,2,0,0.7853981633974483
4,Quratest_0,1.871105e-33,7.256287e-40,0.143542,0.156713,0.160284,0.158812,0.902905,0.903897,True,...,True,True,True,ae,2,Add,cp,2,0,0.7853981633974483
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6715,Quratest_11,2.022463e-97,4.578549e-95,0.221754,0.210048,0.263790,0.258708,0.875223,0.878886,True,...,True,True,True,wstate,4,Replace,y,0,0,
6716,Quratest_12,0.000000e+00,0.000000e+00,0.489750,0.504612,0.575274,0.562925,0.225566,0.249181,True,...,True,True,True,wstate,4,Replace,y,0,0,
6717,Quratest_13,0.000000e+00,0.000000e+00,0.532954,0.514683,0.644678,0.631818,0.394195,0.412624,True,...,True,True,True,wstate,4,Replace,y,0,0,
6718,Quratest_14,0.000000e+00,0.000000e+00,0.270925,0.298716,0.361026,0.347733,0.841199,0.849805,True,...,True,True,True,wstate,4,Replace,y,0,0,


In [2]:
# List of columns related to "Killed" metrics
killed_columns = [col for col in df.columns if col.startswith('Killed_')]

# Calculate the percentage of True values for each "Killed" column
true_percentages = (df[killed_columns].mean() * 100).sort_values(ascending=False)

# Print the results
print("Percentage of True values for each 'Killed' column:")
print(true_percentages)

Percentage of True values for each 'Killed' column:
Killed_NH    95.029762
Killed_NF    94.181548
Killed_NC    93.750000
Killed_NT    92.604167
Killed_IF    90.372024
Killed_IH    86.205357
Killed_IC    85.476190
Killed_IT    81.339286
dtype: float64


In [3]:
def confusionMatrix(col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [4]:
confusion_matrix_chisquare = confusionMatrix('Killed_IC','Killed_NC')
confusion_matrix_hellinger = confusionMatrix('Killed_IH','Killed_NH')
confusion_matrix_trace = confusionMatrix('Killed_IT','Killed_NT')
confusion_matrix_fidelity = confusionMatrix('Killed_IF','Killed_NF')

In [20]:

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Chisquare', 'Hellinger', 'Trace', 'Fidelity'), x_title='Noisy', y_title='Ideal', horizontal_spacing=0.15,vertical_spacing=0.1)
# Define a function to create a heatmap with annotations
def create_heatmap(data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale='Dense',
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=12)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)

# Add heatmaps to subplots
create_heatmap(confusion_matrix_chisquare, row=1, col=1, showscale=True)
create_heatmap(confusion_matrix_hellinger, row=1, col=2, showscale=False)
create_heatmap(confusion_matrix_trace, row=2, col=1, showscale=False)
create_heatmap(confusion_matrix_fidelity, row=2, col=2, showscale=False)

fig.update_layout(
    title_text='Confusion matrix',
    height=600,
    width=600,
    showlegend=False
)

fig.show()